In [ ]:
import gymnasium as gym
env = gym.make("CartPole-v1")


In [ ]:
print(env.observation_space)  # Box(4,) — 4 continuous values (position, velocity, angle, angular velocity)
print(env.action_space)  

In [ ]:
obs, info = env.reset()          # start a new episode

while True:
    action = env.action_space.sample()          # pick an action
    obs, reward, terminated, truncated, info = env.step(action)  # take it
    
    if terminated or truncated:  # pole fell, or 500 steps reached
        break

In [ ]:
import numpy as np

def discretize(obs, bins=10):
    # Define the range for each of the 4 values
    bounds_low  = [-2.4, -3.0, -0.25, -3.0]
    bounds_high = [ 2.4,  3.0,  0.25,  3.0]
    
    ratios = [(obs[i] - bounds_low[i]) / (bounds_high[i] - bounds_low[i]) 
              for i in range(len(obs))]
    
    # Snap each value to one of 10 buckets
    discrete = [int(round(ratio * (bins - 1))) for ratio in ratios]
    discrete = [min(bins - 1, max(0, d)) for d in discrete]  # clamp
    
    return tuple(discrete)  # e.g. (3, 7, 4, 5) — now usable as a dict key

In [ ]:
from collections import defaultdict

class CartPoleAgent:
    def __init__(self, env, learning_rate=0.1, discount=0.99,a
                 epsilon=1.0, epsilon_decay=0.001, min_epsilon=0.01):
        
        self.env = env
        self.q_table = defaultdict(lambda: np.zeros(env.action_space.n))
        
        self.lr = learning_rate
        self.discount = discount
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.min_epsilon = min_epsilon

    def get_action(self, state):
        if np.random.random() < self.epsilon:
            return self.env.action_space.sample()  # explore
        return int(np.argmax(self.q_table[state]))  # exploit

    def update(self, state, action, reward, next_state, terminated):
        future = (not terminated) * np.max(self.q_table[next_state])
        target = reward + self.discount * future
        error  = target - self.q_table[state][action]
        self.q_table[state][action] += self.lr * error

    def decay_epsilon(self):
        self.epsilon = max(self.min_epsilon, self.epsilon - self.epsilon_decay)

In [ ]:
env = gym.make("CartPole-v1")
agent = CartPoleAgent(env)

n_episodes = 10_000
rewards_per_episode = []

for episode in range(n_episodes):
    obs, _ = env.reset()
    state = discretize(obs)         # convert to discrete tuple
    
    total_reward = 0
    
    while True:
        action = agent.get_action(state)
        
        next_obs, reward, terminated, truncated, _ = env.step(action)
        next_state = discretize(next_obs)
        
        agent.update(state, action, reward, next_state, terminated)
        
        state = next_state
        total_reward += reward
        
        if terminated or truncated:
            break
    
    agent.decay_epsilon()
    rewards_per_episode.append(total_reward)
    
    # Print progress every 1000 episodes
    if (episode + 1) % 1000 == 0:
        avg = np.mean(rewards_per_episode[-1000:])
        print(f"Episode {episode+1} | Avg reward: {avg:.1f} | Epsilon: {agent.epsilon:.3f}")

In [ ]:
env_render = gym.make("CartPole-v1", render_mode="human")  # opens a window

obs, _ = env_render.reset()
state = discretize(obs)
total_reward = 0

while True:
    action = agent.get_action(state)   # no exploration — pure exploitation
    obs, reward, terminated, truncated, _ = env_render.step(action)
    state = discretize(obs)
    total_reward += reward
    
    if terminated or truncated:
        break

print(f"Final score: {total_reward}")
env_render.close()